**Environment Setup**

In [16]:
from pathlib import Path
import os

# Define project root (change only this if project location changes)
PROJECT_ROOT = Path("/home/laksh/AI_WORKSPACE/GEN_AI/questionAnswering_system")

# Change working directory to project root
os.chdir(PROJECT_ROOT)

# Verify current working directory
CURRENT_DIR = Path.cwd()
print(f"Working Directory set to: {CURRENT_DIR}")



Working Directory set to: /home/laksh/AI_WORKSPACE/GEN_AI/questionAnswering_system


In [17]:
%pwd

'/home/laksh/AI_WORKSPACE/GEN_AI/questionAnswering_system'

In [18]:
import os
from dotenv import load_dotenv

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY


**Document Loading** 

In [ ]:
# What: Load raw files into Document objects (you already use PyPDFLoader).
# Why: Standardizes content and metadata for downstream processing.
# Tip: If PDFs are scanned, add OCR (Tesseract) first.

'What: Load raw files into Document objects (you already use PyPDFLoader).Why: Standardizes content and metadata for downstream processing.Tip: If PDFs are scanned, add OCR (Tesseract) first.'

In [19]:
from langchain_classic.document_loaders import PyPDFLoader

In [20]:
%pwd

'/home/laksh/AI_WORKSPACE/GEN_AI/questionAnswering_system'

In [21]:
file_path= "data/SDG.pdf"
loader=PyPDFLoader(file_path)
data=loader.load()

In [22]:
data[:10]

[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'creationdate': '2017-12-26T16:03:25-05:00', 'moddate': '2017-12-27T15:29:10-05:00', 'trapped': '/False', 'source': 'data/SDG.pdf', 'total_pages': 24, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'creationdate': '2017-12-26T16:03:25-05:00', 'moddate': '2017-12-27T15:29:10-05:00', 'trapped': '/False', 'source': 'data/SDG.pdf', 'total_pages': 24, 'page': 1, 'page_label': '2'}, page_content=''),
 Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Macintosh)', 'creationdate': '2017-12-26T16:03:25-05:00', 'moddate': '2017-12-27T15:29:10-05:00', 'trapped': '/False', 'source': 'data/SDG.pdf', 'total_pages': 24, 'page': 2, 'page_label': '3'}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE T

In [23]:
len(data)

24

**Text Preprocessing**

In [ ]:
#What: Remove headers/footers, fix whitespace, unify newlines, strip non‑text noise.
#Why: Cleaner text → better chunks and embeddings → fewer hallucinations.

In [24]:
# Combine all page contents into a single string
question_gen=""
for page in data:
    question_gen += page.page_content

In [25]:
question_gen

'IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence that we can succeed. In the past 15 yea

**Chunking & Splitting**

In [ ]:
# What: Break long text into manageable chunks (token-based).
# Recommended: 800–1500 tokens per chunk, 10–20% overlap (e.g., chunk_size=1000, overlap=100).
# Why: Keeps each chunk within model context and preserves cross‑chunk continuity.

In [26]:
# Create a text splitter instance
from langchain_classic.text_splitter import TokenTextSplitter

splitter_qiuz= TokenTextSplitter(
    model_name="gpt-3.5-turbo",
    chunk_size=10000,
    chunk_overlap=200,
)

In [27]:
# Split the combined text into chunks
chunk_quiz = splitter_qiuz.split_text(question_gen)

In [28]:
chunk_quiz

['IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence that we can succeed. In the past 15 ye

In [29]:
len(chunk_quiz)

1

In [30]:
# type(chunk_quiz)
type(chunk_quiz[0])

str

In [31]:
# Create Document objects from the text chunks
from langchain_classic.docstore.document import Document


documents_quiz = [Document(page_content=chunk) for chunk in chunk_quiz]
documents_quiz

[Document(metadata={}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence tha

In [32]:
type(documents_quiz[0])

langchain_core.documents.base.Document

In [33]:

# Create another text splitter instance for answers
splitter_answer= TokenTextSplitter(
    model_name="gpt-3.5-turbo",
    chunk_size=1000,
    chunk_overlap=100,
)

In [34]:
documents_answer= splitter_answer.split_documents(documents_quiz) # Split the documents into smaller chunks for answers

In [35]:
documents_answer # Split the documents into smaller chunks for answers

[Document(metadata={}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence tha

In [36]:
len(documents_answer)

4

**LLM Initialization**

In [37]:
# Create LLM instance for question generation, that generates questions from the content
from langchain_google_genai import ChatGoogleGenerativeAI

llm_ques_gen_pipeline = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.3
)

**Prompt Engineering**

In [38]:
prompt_template = """
You are an expert at creating questions based on coding materials and documentation.
Your goal is to prepare a coder or programmer for their exam and coding tests.
You do this by asking questions about the text below:
----------------
{text}
----------------

Create questions that will prepare the coders or programmers for the
Make sure not to lose any important information.

QUESTIONS:
"""

In [39]:
# Create PromptTemplate instance, that will be used to generate questions
from langchain_classic.prompts import PromptTemplate

prompt_quiz = PromptTemplate(
    input_variables=["text"],
    template=prompt_template)

In [40]:
# Define the refine template, which will help to improve the questions

refine_template = ("""
You are an expert at creating practice questions based on coding material and documentation.
Your goal is to help a coder or programmer prepare for a coding test.
We have received some practice questions to a certain extent: {existing_answer}.
We have the option to refine the existing questions or add new ones
(only if necessary) with some more context below.

{text}
------------

Given the new context, refine the original questions in English.
If the context is not helpful, please provide the original questions.

QUESTIONS:
""")


In [41]:
# Create PromptTemplate instance for refining questions
refine_question = PromptTemplate(
    input_variables=["existing_answer", "text"],
    template=refine_template)

**Question generation chain**

In [ ]:
# You already use: load_summarize_chain with chain_type="refine" — perfect for generating/refining practice questions across chunks.
# Tip: Keep a deterministic low temperature for consistent question quality.

In [42]:
# Create the question generation chain using summarize chain with refine method, that generates questions

from langchain_classic.chains.summarize import load_summarize_chain

ques_gen_chain = load_summarize_chain(llm = llm_ques_gen_pipeline, 
                                          chain_type = "refine", 
                                          verbose = True, 
                                          question_prompt=prompt_quiz, 
                                          refine_prompt=refine_question)

In [43]:
# Generate questions from the documents
ques = ques_gen_chain.run(documents_quiz)

print(ques)

/tmp/ipykernel_35637/3342295842.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  ques = ques_gen_chain.run(documents_quiz)




> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

You are an expert at creating questions based on coding materials and documentation.
Your goal is to prepare a coder or programmer for their exam and coding tests.
You do this by asking questions about the text below:
----------------
IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD 
CAME TOGETHER TO FACE THE FUTURE.
And what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. 
Not just in some faraway place, but in their own cities and towns and villages.
They knew things didn’t have to be this way. They knew we had enough 
food to feed the world, but that it wasn’t getting shared. They knew there 
were medicines for HIV and other diseases, but they cost a lot. They knew 
that earthquakes and floods were inevitable, but that the high death 
tolls were not. 
They also knew that billions of people worldwide shared their hope for a 
better future.
So leaders f

**Embeddings Model**

In [ ]:
# Options: FAISS (local, fast) for prototyping; Pinecone/Milvus/Weaviate for production.
# Recommendation: Start with FAISS (persist the index for reuse).
# Why: Fast similarity search with local control.

In [44]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

**Create Vector Store using FAISS for RAG**

In [ ]:
# Options: FAISS (local, fast) for prototyping; Pinecone/Milvus/Weaviate for production.
# Recommendation: Start with FAISS (persist the index for reuse).
# Why: Fast similarity search with local control.

In [45]:
# Create FAISS vector store from documents
from langchain_community.vectorstores import FAISS

# Create vector store using documents_answer and embeddings
vector_store = FAISS.from_documents(
    documents=documents_answer,
    embedding=embeddings
)

print(f"Vector store created with {len(documents_answer)} documents")

Vector store created with 4 documents


**Create Retriever from Vector Store**

In [ ]:
# What: Given a query, return top-k relevant chunks.
# Settings: k=3–6 is a good starting point; tune by evaluation.
# Advanced: Use hybrid retrieval (BM25 lexical + dense vectors) or a cross-encoder re-ranker for better precision.

In [46]:
# Create a retriever from the vector store
# retriever will search for k most relevant documents based on similarity
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Retrieve top 3 most relevant documents
)

print("Retriever created successfully")

Retriever created successfully


**Create LLM instance for answer generation**

In [ ]:
# Chain types:
# stuff — concatenates docs into one prompt (simple, risky for long contexts).
# map_reduce — runs LLM on smaller groups then aggregates (best for long sources).
# refine — iteratively improves an answer (good for synthesis or question generation).
# Recommendation for this project: Use map_reduce or refine for robust answers from long PDFs; use stuff only when context is small.
# LLM settings: low temperature (0.0–0.3) for factual answers; keep verbose=False for production.
# Prompting: include an instruction asking for citations and short answers.

In [47]:
# Create LLM instance for answer generation
llm_answer_gen = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.7
)

**Create RAG Chain (Retrieval-Augmented Generation)**

In [48]:
ques

'Here are some questions based on the provided text, designed to prepare coders and programmers for exams and coding tests. These questions focus on extracting specific information, understanding relationships, and identifying key details that might be relevant in a technical context.\n\n**General Knowledge & Context:**\n\n1.  What year did leaders from 193 countries come together to address global challenges, leading to the creation of the Sustainable Development Goals (SDGs)?\n2.  What is the primary overarching vision of the Sustainable Development Goals (SDGs) for the year 2030?\n3.  Which organization is highlighted as a leading entity working to fulfill the SDGs, and in how many countries is it present?\n4.  What is the target year for achieving the Sustainable Development Goals?\n5.  The text mentions several global challenges that leaders recognized. List at least five of these challenges.\n\n**Specific Goal-Oriented Questions:**\n\n6.  **Goal: End Extreme Poverty:**\n    *   W

In [49]:
ques_list= ques.split("\n")

In [50]:
ques_list=ques_list[:10]  # Display first 10 questions
print(ques_list)

['Here are some questions based on the provided text, designed to prepare coders and programmers for exams and coding tests. These questions focus on extracting specific information, understanding relationships, and identifying key details that might be relevant in a technical context.', '', '**General Knowledge & Context:**', '', '1.  What year did leaders from 193 countries come together to address global challenges, leading to the creation of the Sustainable Development Goals (SDGs)?', '2.  What is the primary overarching vision of the Sustainable Development Goals (SDGs) for the year 2030?', '3.  Which organization is highlighted as a leading entity working to fulfill the SDGs, and in how many countries is it present?', '4.  What is the target year for achieving the Sustainable Development Goals?', '5.  The text mentions several global challenges that leaders recognized. List at least five of these challenges.', '']


In [51]:
from langchain_classic.chains import RetrievalQA

answer_generation_chain = RetrievalQA.from_chain_type(llm=llm_answer_gen, 
                                               chain_type="stuff", 
                                               retriever=vector_store.as_retriever())

**Test RAG Pipeline with Sample Questions**

In [ ]:
# What: Split documents into smaller chunks for embedding and retrieval.
# Why: LLMs have context limits; smaller chunks improve relevance and reduce hallucinations.
# Tip: Use token-based splitting (e.g., 1000 tokens with 10-20% overlap) for best results.

In [52]:
# Test the RAG pipeline with a sample question
sample_question = "What are the Sustainable Development Goals?"

print(f"Question: {sample_question}")
print("\n" + "="*80)

# Invoke the RAG chain (use the expected input key 'query')
response = answer_generation_chain.invoke({"query": sample_question})

print("\nAnswer:")
print(response)
print("\n" + "="*80 + "\n")

# Display the source documents used
print("Source Documents Used:")
for i, doc in enumerate(response.get("context", []), 1):
    print(f"\nDocument {i}:")
    print(doc.page_content[:500] + "..." if len(doc.page_content) > 500 else doc.page_content)

Question: What are the Sustainable Development Goals?


Answer:
{'query': 'What are the Sustainable Development Goals?', 'result': 'The Sustainable Development Goals (SDGs) are a plan created by leaders from 193 countries in 2015. This set of 17 goals aims to create a future, by 2030, that is free from poverty and hunger, and protected from the worst effects of climate change.'}


Source Documents Used:


In [53]:

for question in ques_list:
    # Skip empty questions
    if not question or not question.strip():
        continue
    
    print("Question: ", question)
    answer = answer_generation_chain.run(question)
    print("Answer: ", answer)
    print("--------------------------------------------------\n\n")
    # Save answer to file
    with open("answers.txt", "a") as f:
        f.write("Question: " + question + "\n")
        f.write("Answer: " + answer + "\n")
        f.write("--------------------------------------------------\n\n")

Question:  Here are some questions based on the provided text, designed to prepare coders and programmers for exams and coding tests. These questions focus on extracting specific information, understanding relationships, and identifying key details that might be relevant in a technical context.
Answer:  Okay, I'm ready! Please provide me with the questions. I will do my best to answer them based on the text you've provided, keeping in mind the context of preparing coders and programmers for exams and coding tests.
--------------------------------------------------


Question:  **General Knowledge & Context:**
Answer:  The provided text discusses the **Sustainable Development Goals (SDGs)**, a set of 17 ambitious goals established in 2015 by leaders from 193 countries. These goals aim to create a better future by addressing global challenges such as poverty, hunger, inequality, climate change, and environmental degradation, with a target year of 2030.

The text highlights progress made 

**Question Answering Function**

In [56]:
# Create a reusable function for question answering
def answer_question(question, show_sources=True):
    """
    Answer a question using the RAG pipeline
    
    Args:
        question (str): The question to answer
        show_sources (bool): Whether to display source documents
        
    Returns:
        dict: Contains 'answer' (str) and 'context' (list of source docs)
    """
    # RetrievalQA expects the input under the "query" key when using invoke
    result = answer_generation_chain.invoke({"query": question})
    
    # Normalize answer text (support different returned shapes)
    if isinstance(result, dict):
        answer_text = result.get("result") or result.get("answer") or result.get("output") or result.get("text") or ""
    else:
        answer_text = str(result)
    
    print(f"\n{'='*80}")
    print(f"Question: {question}")
    print(f"{'='*80}")
    print(f"\nAnswer:\n{answer_text}")
    
    # Extract source documents from common keys
    sources = []
    if isinstance(result, dict):
        sources = result.get("context") or result.get("source_documents") or result.get("sources") or []
    
    if show_sources and sources:
        print(f"\n{'='*80}")
        print("Source Documents:")
        for i, doc in enumerate(sources, 1):
            # Support Document objects, dicts with 'page_content', or raw strings
            if hasattr(doc, "page_content"):
                content = doc.page_content
            elif isinstance(doc, dict) and "page_content" in doc:
                content = doc["page_content"]
            else:
                content = str(doc)
            content_preview = content[:300] + "..." if len(content) > 300 else content
            print(f"\nDocument {i}:")
            print(content_preview)
    
    print(f"\n{'='*80}\n")
    return {"answer": answer_text, "context": sources}

# Test the function with a few sample questions
test_questions = [
    "What are SDGs?",
    "How many Sustainable Development Goals are there?",
]

for q in test_questions:
    answer_question(q, show_sources=False)


Question: What are SDGs?

Answer:
SDGs stands for the **Sustainable Development Goals**.

These are a set of 17 goals created by leaders from 193 countries in 2015 with the aim of addressing global challenges and creating a better future by the year 2030. The plan envisions a future that is rid of poverty and hunger, and safe from the worst effects of climate change.



Question: How many Sustainable Development Goals are there?

Answer:
There are 17 Sustainable Development Goals.


